# Notebook 05: S3-Trigger CI/CD Pipeline (Simplified, Jupyter Monitoring Loop)

Module: ITI113 Machine Learning & Operations

Focus Area: C - MLOps & Deployment

Estimated Runtime: 10-20 minutes (Section 4 waits for a full pipeline execution)

---

### What this notebook does

Builds a triggered version of Notebook 03's SageMaker Pipeline. Instead of a hardcoded dataset path, the training data location is a runtime parameter (InputDataUrl), and the pipeline starts automatically when a new CSV lands in a watched S3 prefix, a minimal stand-in for a production CI/CD retraining trigger.

1. Rewrites preprocess.py, train.py, and inference.py (unchanged from Notebook 03) and uploads them to S3
2. Defines the triggered pipeline: PreprocessData -> TrainModel -> AUCQualityGate -> RegisterModel (registration only runs if the quality gate passes)
3. Simulates the S3 trigger end-to-end: detects a new file and starts a pipeline execution automatically
4. Monitors the execution to completion and confirms model registration
5. Maps each mechanism here to its production CI/CD equivalent (EventBridge, Lambda, CloudWatch)

Same four-step shape as Notebook 03's manual pipeline, so the two are easy to compare side by side. The only real difference is a runtime parameter instead of a hardcoded path.


## 0. Configuration

Sets up the SageMaker session, S3 bucket and prefix, the manual and
triggered pipeline names, the S3 trigger watch folder, and the quality
gate threshold used throughout this notebook.


In [1]:
%%capture
%pip install --upgrade "sagemaker>=2,<3" boto3 botocore
%pip install --upgrade mlflow sagemaker-mlflow
print("Packages installed.")

In [2]:
import os
os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("sagemaker").setLevel(logging.ERROR)

import boto3
import sagemaker
import json
import time
import glob
from pathlib import Path
from datetime import datetime

session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

BUCKET = "nyp-26s1-iti113"

TEAM_ID = "team03"
STUDENT_ID = "s301"

COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "crypto-scam-detector"

PREFIX = f"iti113/{TEAM_ID}/data/{PROJECT_NAME}"

PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE = "ml.m5.large"

# Manual pipeline and raw dataset, used as the default InputDataUrl when nothing has triggered this pipeline yet.
MANUAL_PIPELINE_NAME = f"iti113-{TEAM_ID}-crypto-scam-detector"
RAW_DATA_URI = f"s3://{BUCKET}/{PREFIX}/raw/crypto_scam_dataset.csv"

# The triggered pipeline is a separate pipeline object that shares the same Model Registry group, so manual and triggered runs both version into one registry.
TRIGGERED_PIPELINE_NAME = f"iti113-{TEAM_ID}-crypto-scam-detector-triggered"
MODEL_PACKAGE_GROUP = f"{TEAM_ID}-CryptoScamDetector"
QUALITY_GATE_AUC = 0.85

PIPELINE_ROOT = f"s3://{BUCKET}/{PREFIX}/pipeline-triggered"

SCRIPTS_S3_PREFIX = f"{PREFIX}/pipeline_src"
SCRIPTS_S3_URI = f"s3://{BUCKET}/{SCRIPTS_S3_PREFIX}"
LOCAL_PIPELINE_SRC = "pipeline_src"

# S3-trigger watch folder, stand-in for the production EventBridge rule.
TRIGGER_PREFIX = f"{PREFIX}/trigger"
TRIGGER_INCOMING_PREFIX = f"{TRIGGER_PREFIX}/incoming"
TRIGGER_INCOMING_URI = f"s3://{BUCKET}/{TRIGGER_INCOMING_PREFIX}"
SEEN_STATE_FILE = Path(f"trigger_seen_keys_{TEAM_ID}.json")

print(
    f"Manual pipeline: {MANUAL_PIPELINE_NAME}\n"
    f"Triggered pipeline: {TRIGGERED_PIPELINE_NAME}\n"
    f"Model Registry group: {MODEL_PACKAGE_GROUP}\n"
    f"Bucket: {BUCKET}\n"
    f"Team prefix: {PREFIX}\n"
    f"Region: {region}\n"
    f"SageMaker role: {role}\n"
    f"Pipeline source S3 URI: {SCRIPTS_S3_URI}\n"
    f"Trigger watch folder: {TRIGGER_INCOMING_URI}"
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


Manual pipeline: iti113-team03-crypto-scam-detector
Triggered pipeline: iti113-team03-crypto-scam-detector-triggered
Model Registry group: team03-CryptoScamDetector
Bucket: nyp-26s1-iti113
Team prefix: iti113/team03/data/crypto-scam-detector
Region: ap-southeast-1
SageMaker role: arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team03
Pipeline source S3 URI: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src
Trigger watch folder: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming


## 1. Pipeline Scripts

`preprocess.py` and `inference.py` are identical to Notebook 03's, so the triggered
pipeline computes exactly the same engineered features as the manual one and models
registered by either are directly comparable. `train.py` is a reduced form: Notebook 03's
version supports both Logistic Regression and Random Forest so it can train whichever
model won the comparison, whereas this triggered pipeline retrains the winning
architecture, Logistic Regression only. This notebook writes its own copies so it can
run standalone without Notebook 03 having executed first in the same kernel.

In [3]:
os.makedirs('src', exist_ok=True)
print('src/ directory ready')

src/ directory ready


In [4]:
%%writefile src/preprocess.py
"""SageMaker Processing Job for text preprocessing.

Cleans the raw scam-message text, builds engineered indicator features
(urgency, contact/link, structural characteristics), fits TF-IDF on the
training split only, and saves what training and serving need to stay in
sync: TF-IDF features (.npz), engineered features and labels (CSV), and a
preprocessor bundle (joblib) with the fitted vectorizer.
"""
import os
import re
import argparse
import glob

import pandas as pd
import numpy as np
import joblib
import scipy.sparse as sp

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import subprocess
import sys

try:
    from rapidfuzz import fuzz
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])
    from rapidfuzz import fuzz

parser = argparse.ArgumentParser()
parser.add_argument('--test-size',    type=float, default=0.20)
parser.add_argument('--random-state', type=int,   default=42)
args = parser.parse_args()

# Discovers the input CSV by filename pattern rather than hardcoding the dataset name
input_dir = '/opt/ml/processing/input'
csv_candidates = sorted(glob.glob(os.path.join(input_dir, '*.csv')))
if not csv_candidates:
    raise FileNotFoundError(
        f'No CSV file found in {input_dir}. Expected the file that triggered '
        'this pipeline run (or crypto_scam_dataset.csv for a manual run).'
    )
input_path = csv_candidates[0]
print(f'Using input file: {input_path}')
output_dir = '/opt/ml/processing/output'
os.makedirs(output_dir, exist_ok=True)

SCAM_LABEL = "scam"

# Text cleaning, mirrors utils/preprocessing.py's clean_text()
URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Engineered indicator features, keyword lists follow the project proposal's indicator design
URGENT_KEYWORDS = [
    "urgent", "immediately", "hurry", "act now", "act fast", "limited time",
    "limited slots", "limited spots", "today only", "last chance",
    "final notice", "final warning", "don't miss", "don't wait",
    "expires", "expiring", "closing soon", "ending soon", "before it's too late",
    "respond now", "reply now", "confirm now", "verify now", "claim now",
    "while supplies last", "only a few left", "act before", "time-sensitive",
    "this offer", "exclusive offer", "one time offer",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed", "guarantee", "risk-free", "risk free", "no risk",
    "zero risk", "sure profit", "sure win", "can't lose", "cannot lose",
    "double your money", "triple your money", "multiply your", "10x", "100x",
    "passive income", "easy money", "get rich", "financial freedom",
    "life-changing", "once in a lifetime", "secret method", "proven strategy",
    "insider information", "insider tip", "exclusive access", "vip access",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer", "send funds", "send payment", "send money",
    "top up", "top-up", "processing fee", "activation fee", "unlock fee",
    "advance fee", "small fee", "gas fee", "network fee", "wallet",
    "crypto", "bitcoin", "ethereum", "usdt", "stablecoin",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "signal", "wechat", "line app",
    "private chat", "private message", "dm me", "direct message",
    "text me", "call me", "contact me directly", "reach out privately",
    "add me on",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "recovery phrase", "recovery key",
    "wallet password", "login credentials", "verification code", "otp",
    "security code", "pin number", "social security"
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches_fuzzy(message, keywords, threshold=85):
    message_lower = message.lower()
    tokens = message_lower.split()
    matches = []
    for kw in keywords:
        kw_lower = kw.lower()
        if " " in kw_lower:
            if fuzz.partial_ratio(kw_lower, message_lower) >= threshold:
                matches.append(kw)
        else:
            if any(fuzz.ratio(kw_lower, tok) >= threshold for tok in tokens):
                matches.append(kw)
    return matches

def extract_engineered_features(text):
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches_fuzzy(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches_fuzzy(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches_fuzzy(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches_fuzzy(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches_fuzzy(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
    }

ENGINEERED_COLS = [
    "urgency_keyword_count", "guaranteed_return_keyword_count", "countdown_phrase_count",
    "exclamation_count", "urgency_score", "has_wallet_address",
    "has_url", "url_count", "has_email", "has_phone_number", "payment_keyword_count",
    "off_platform_keyword_count", "credential_keyword_count",
    "capital_letter_ratio", "has_numeric_content",
]

# Load, clean, engineer, split
df = pd.read_csv(input_path)
expected_columns = {"id", "platform", "text", "label"}
missing = expected_columns - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing expected columns: {missing}")

df["clean_text"] = df["text"].apply(clean_text)

engineered = pd.DataFrame([extract_engineered_features(t) for t in df["text"]], index=df.index)
df = pd.concat([df, engineered], axis=1)

train_df, test_df = train_test_split(
    df,
    test_size=args.test_size,
    random_state=args.random_state,
    stratify=df["label"],
)

print(f"Train: {train_df.shape[0]} rows | Test: {test_df.shape[0]} rows")

# TF-IDF, fit on train text only, never on test
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
vectorizer.fit(train_df["clean_text"])

X_train_tfidf = vectorizer.transform(train_df["clean_text"])
X_test_tfidf = vectorizer.transform(test_df["clean_text"])

y_train = (train_df["label"] == SCAM_LABEL).astype(int)
y_test = (test_df["label"] == SCAM_LABEL).astype(int)

print(f"TF-IDF vocabulary size: {len(vectorizer.get_feature_names_out())}")

# Save TF-IDF as sparse .npz (a dense CSV would be hundreds of MB); everything else as CSV
sp.save_npz(f"{output_dir}/train_tfidf.npz", X_train_tfidf)
sp.save_npz(f"{output_dir}/test_tfidf.npz", X_test_tfidf)

train_df[ENGINEERED_COLS].to_csv(f"{output_dir}/train_engineered.csv", index=False)
test_df[ENGINEERED_COLS].to_csv(f"{output_dir}/test_engineered.csv", index=False)

y_train.to_csv(f"{output_dir}/train_labels.csv", index=False, header=True)
y_test.to_csv(f"{output_dir}/test_labels.csv", index=False, header=True)

preprocessor = {
    "vectorizer": vectorizer,
    "engineered_cols": ENGINEERED_COLS,
    "feature_columns": list(vectorizer.get_feature_names_out()) + ENGINEERED_COLS,
}
joblib.dump(preprocessor, f"{output_dir}/preprocessor.joblib")

print("Preprocessing complete. Saved TF-IDF (.npz), engineered features, labels, and preprocessor.joblib.")
print(f"Final feature count: {len(preprocessor['feature_columns'])}")

Overwriting src/preprocess.py


In [5]:
%%writefile src/train.py
"""SageMaker Training Job.

Trains Logistic Regression on TF-IDF + engineered features and saves a
deployment bundle. Chosen over Random Forest as the strongest candidate
from experimentation (test AUC-ROC 0.8908 versus 0.86), clearing the 0.85
quality gate with a comfortable margin. Defaults to the winning config
(C=10.0, penalty=l2, solver=lbfgs, max_iter=1000), overridable as pipeline
parameters.

The bundle carries both the trained model and the fitted TF-IDF vectorizer
so the endpoint can accept raw text and reproduce training-time features.
MLflow logging happens outside this container, in the notebook, after a
successful run.
"""
import os
import argparse
import pickle
import joblib
import pandas as pd
import scipy.sparse as sp

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
)

parser = argparse.ArgumentParser()
parser.add_argument('--reg-c', type=float, default=10.0)
parser.add_argument('--penalty', type=str, default='l2')
parser.add_argument('--solver', type=str, default='lbfgs')
parser.add_argument('--max-iter', type=int, default=1000)
parser.add_argument('--random-state', type=int, default=42)

parser.add_argument('--team-id', type=str, default=os.environ.get('TEAM_ID', 'unknown-team'))
parser.add_argument('--student-id', type=str, default=os.environ.get('STUDENT_ID', 's000'))
parser.add_argument('--semester', type=str, default=os.environ.get('SEMESTER', '26S1'))
parser.add_argument('--run-name', type=str, default='sagemaker_pipeline_run')

parser.add_argument(
    '--model-dir',
    type=str,
    default=os.environ.get('SM_MODEL_DIR', '/opt/ml/model')
)
parser.add_argument(
    '--train',
    type=str,
    default=os.environ.get('SM_CHANNEL_TRAIN', '/opt/ml/input/data/train')
)
parser.add_argument(
    '--test',
    type=str,
    default=os.environ.get('SM_CHANNEL_TEST', '/opt/ml/input/data/test')
)
args = parser.parse_args()

os.makedirs(args.model_dir, exist_ok=True)

print(
    f'SageMaker Training Environment:\n'
    f'Train channel: {args.train}\n'
    f'Test channel: {args.test}\n'
    f'Model directory: {args.model_dir}'
)

X_train_tfidf = sp.load_npz(os.path.join(args.train, 'train_tfidf.npz'))
X_test_tfidf = sp.load_npz(os.path.join(args.test, 'test_tfidf.npz'))

train_engineered = pd.read_csv(os.path.join(args.train, 'train_engineered.csv'))
test_engineered = pd.read_csv(os.path.join(args.test, 'test_engineered.csv'))

y_train = pd.read_csv(os.path.join(args.train, 'train_labels.csv')).squeeze('columns')
y_test = pd.read_csv(os.path.join(args.test, 'test_labels.csv')).squeeze('columns')

preprocessor_path = os.path.join(args.train, 'preprocessor.joblib')
if not os.path.exists(preprocessor_path):
    raise FileNotFoundError(
        f'preprocessor.joblib was not found at {preprocessor_path}. '
        'Rerun the ProcessingStep with the updated preprocess.py.'
    )

preprocessor = joblib.load(preprocessor_path)
engineered_cols = preprocessor['engineered_cols']

X_train = sp.hstack([X_train_tfidf, train_engineered[engineered_cols].values]).tocsr()
X_test = sp.hstack([X_test_tfidf, test_engineered[engineered_cols].values]).tocsr()

print(f'Train: {X_train.shape[0]} rows, {X_train.shape[1]} features\nTest: {X_test.shape[0]} rows')

if len(pd.Series(y_train).unique()) < 2:
    raise ValueError('Training labels contain fewer than two classes.')

model = LogisticRegression(
    C=args.reg_c,
    penalty=args.penalty,
    solver=args.solver,
    max_iter=args.max_iter,
    class_weight='balanced',
    random_state=args.random_state,
)
model.fit(X_train, y_train)

all_metrics = {}
for split, X, y in [('train', X_train, y_train), ('test', X_test, y_test)]:
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    all_metrics.update({
        f'{split}_accuracy': round(accuracy_score(y, predictions), 4),
        f'{split}_f1': round(f1_score(y, predictions, zero_division=0), 4),
        f'{split}_precision': round(
            precision_score(y, predictions, zero_division=0), 4
        ),
        f'{split}_recall': round(
            recall_score(y, predictions, zero_division=0), 4
        ),
    })

    if len(pd.Series(y).unique()) >= 2:
        all_metrics[f'{split}_auc_roc'] = round(
            roc_auc_score(y, probabilities), 4
        )
    else:
        all_metrics[f'{split}_auc_roc'] = None
        print(f'Warning: {split} split has only one class; AUC-ROC unavailable.')

print('Metrics:')
for metric_name, metric_value in all_metrics.items():
    print(f'{metric_name}: {metric_value}')

model_bundle = {
    'model': model,
    'preprocessor': preprocessor,
    'engineered_cols': engineered_cols,
    'input_format': 'raw_text_json',
    'description': (
        'Deployment bundle containing trained model and fitted TF-IDF vectorizer. '
        'Endpoint accepts raw message text as JSON: {"text": "..."}'
    ),
}

joblib.dump(model_bundle, os.path.join(args.model_dir, 'model.joblib'))

with open(os.path.join(args.model_dir, 'model.pkl'), 'wb') as f:
    pickle.dump(model_bundle, f)

print(
    f"Model bundle saved: {os.path.join(args.model_dir, 'model.joblib')}\n"
    f"Legacy bundle saved: {os.path.join(args.model_dir, 'model.pkl')}"
)

if all_metrics['test_auc_roc'] is None:
    raise ValueError('Test AUC-ROC is unavailable; cannot evaluate the quality gate.')

print(
    f"Test AUC-ROC: {all_metrics['test_auc_roc']}\n"
    f"test_accuracy: {all_metrics['test_accuracy']}\n"
    f"test_f1: {all_metrics['test_f1']}\n"
    f"test_precision: {all_metrics['test_precision']}\n"
    f"test_recall: {all_metrics['test_recall']}"
)

Overwriting src/train.py


In [6]:
%%writefile src/inference.py
"""SageMaker inference handler for the deployed crypto-scam-detector endpoint.

Accepts JSON input with raw message text, e.g. {"text": "..."}, a list of
such records, or {"instances": [...]}. Applies the same text cleaning,
engineered-feature extraction and TF-IDF transform saved in model.joblib
before predicting, matching utils/preprocessing.py, so serving matches
training.
"""
import os
import re
import json
import pickle
import joblib
import pandas as pd
import scipy.sparse as sp

import subprocess
import sys

try:
    from rapidfuzz import fuzz
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"])
    from rapidfuzz import fuzz

URL_PATTERN = re.compile(r"https?://[^\s]+|www\.[^\s]+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = URL_PATTERN.sub(" <url> ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

URGENT_KEYWORDS = [
    "urgent", "immediately", "hurry", "act now", "act fast", "limited time",
    "limited slots", "limited spots", "today only", "last chance",
    "final notice", "final warning", "don't miss", "don't wait",
    "expires", "expiring", "closing soon", "ending soon", "before it's too late",
    "respond now", "reply now", "confirm now", "verify now", "claim now",
    "while supplies last", "only a few left", "act before", "time-sensitive",
    "this offer", "exclusive offer", "one time offer",
]
GUARANTEED_RETURN_KEYWORDS = [
    "guaranteed", "guarantee", "risk-free", "risk free", "no risk",
    "zero risk", "sure profit", "sure win", "can't lose", "cannot lose",
    "double your money", "triple your money", "multiply your", "10x", "100x",
    "passive income", "easy money", "get rich", "financial freedom",
    "life-changing", "once in a lifetime", "secret method", "proven strategy",
    "insider information", "insider tip", "exclusive access", "vip access",
]
PAYMENT_KEYWORDS = [
    "deposit", "transfer", "send funds", "send payment", "send money",
    "top up", "top-up", "processing fee", "activation fee", "unlock fee",
    "advance fee", "small fee", "gas fee", "network fee", "wallet",
    "crypto", "bitcoin", "ethereum", "usdt", "stablecoin",
]
OFF_PLATFORM_KEYWORDS = [
    "telegram", "whatsapp", "discord", "signal", "wechat", "line app",
    "private chat", "private message", "dm me", "direct message",
    "text me", "call me", "contact me directly", "reach out privately",
    "add me on",
]
CREDENTIAL_KEYWORDS = [
    "seed phrase", "private key", "recovery phrase", "recovery key",
    "wallet password", "login credentials", "verification code", "otp",
    "security code", "pin number", "social security"
]

WALLET_PATTERN = re.compile(r"\b(?:0x[a-fA-F0-9]{40}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b")
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_PATTERN = re.compile(r"(?:\+?\d[\d\s-]{7,}\d)")
COUNTDOWN_PATTERN = re.compile(r"\b\d+\s*(?:hour|hours|hr|hrs|minute|minutes|min|mins|day|days)\b", re.IGNORECASE)

def find_keyword_matches_fuzzy(message, keywords, threshold=85):
    message_lower = message.lower()
    tokens = message_lower.split()
    matches = []
    for kw in keywords:
        kw_lower = kw.lower()
        if " " in kw_lower:
            if fuzz.partial_ratio(kw_lower, message_lower) >= threshold:
                matches.append(kw)
        else:
            if any(fuzz.ratio(kw_lower, tok) >= threshold for tok in tokens):
                matches.append(kw)
    return matches

def extract_engineered_features(text):
    if not isinstance(text, str):
        text = ""

    urgency_keyword_hits = len(find_keyword_matches_fuzzy(text, URGENT_KEYWORDS))
    guaranteed_return_hits = len(find_keyword_matches_fuzzy(text, GUARANTEED_RETURN_KEYWORDS))
    countdown_hits = len(COUNTDOWN_PATTERN.findall(text))
    exclamation_count = text.count("!")
    urgency_score = urgency_keyword_hits + countdown_hits + min(exclamation_count, 3)

    wallet_matches = WALLET_PATTERN.findall(text)
    url_matches = URL_PATTERN.findall(text)
    email_matches = EMAIL_PATTERN.findall(text)
    phone_matches = PHONE_PATTERN.findall(text)
    payment_hits = len(find_keyword_matches_fuzzy(text, PAYMENT_KEYWORDS))
    off_platform_hits = len(find_keyword_matches_fuzzy(text, OFF_PLATFORM_KEYWORDS))
    credential_hits = len(find_keyword_matches_fuzzy(text, CREDENTIAL_KEYWORDS))

    letters = [c for c in text if c.isalpha()]
    capital_ratio = sum(1 for c in letters if c.isupper()) / len(letters) if letters else 0.0
    digit_count = sum(1 for c in text if c.isdigit())

    return {
        "urgency_keyword_count": urgency_keyword_hits,
        "guaranteed_return_keyword_count": guaranteed_return_hits,
        "countdown_phrase_count": countdown_hits,
        "exclamation_count": exclamation_count,
        "urgency_score": urgency_score,
        "has_wallet_address": int(bool(wallet_matches)),
        "has_url": int(bool(url_matches)),
        "url_count": len(url_matches),
        "has_email": int(bool(email_matches)),
        "has_phone_number": int(bool(phone_matches)),
        "payment_keyword_count": payment_hits,
        "off_platform_keyword_count": off_platform_hits,
        "credential_keyword_count": credential_hits,
        "capital_letter_ratio": round(capital_ratio, 4),
        "has_numeric_content": int(digit_count > 0),
    }


def model_fn(model_dir):
    """Load the model bundle from the SageMaker model directory."""
    joblib_path = os.path.join(model_dir, 'model.joblib')
    pkl_path = os.path.join(model_dir, 'model.pkl')

    if os.path.exists(joblib_path):
        bundle = joblib.load(joblib_path)
    elif os.path.exists(pkl_path):
        with open(pkl_path, 'rb') as f:
            bundle = pickle.load(f)
    else:
        raise FileNotFoundError('Neither model.joblib nor model.pkl was found.')

    return bundle


def input_fn(body, content_type='application/json'):
    """Parse JSON request body into a DataFrame of raw message-text records."""
    if content_type != 'application/json':
        raise ValueError(f'Unsupported content type: {content_type}')

    payload = json.loads(body)

    if isinstance(payload, dict):
        if 'instances' in payload:
            payload = payload['instances']
        else:
            payload = [payload]

    if not isinstance(payload, list):
        raise ValueError('JSON input must be a dictionary, a list of dictionaries, or {"instances": [...]}')

    return pd.DataFrame(payload)


def predict_fn(data, bundle):
    """Clean text, extract engineered features, TF-IDF transform, then predict."""
    if 'text' not in data.columns:
        raise ValueError('Input must include a "text" field with the raw message.')

    model = bundle['model']
    preprocessor = bundle['preprocessor']
    vectorizer = preprocessor['vectorizer']
    engineered_cols = preprocessor['engineered_cols']

    clean = data['text'].apply(clean_text)
    engineered = pd.DataFrame(
        [extract_engineered_features(t) for t in data['text']], index=data.index
    )

    X_tfidf = vectorizer.transform(clean)
    X = sp.hstack([X_tfidf, engineered[engineered_cols].values]).tocsr()

    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    return predictions, probabilities


def output_fn(prediction, accept='application/json'):
    preds, probas = prediction
    response = [
        {
            'prediction': int(p),
            'label': 'Scam' if int(p) == 1 else 'Legitimate',
            'probability': round(float(b), 4),
        }
        for p, b in zip(preds, probas)
    ]
    return json.dumps(response), accept

Overwriting src/inference.py


In [7]:
print("Scripts written:")
for fn in ["preprocess.py", "train.py", "inference.py"]:
    size = os.path.getsize(f"src/{fn}")
    print(f"  src/{fn}  ({size} bytes)")

Scripts written:
  src/preprocess.py  (8966 bytes)
  src/train.py  (5704 bytes)
  src/inference.py  (7725 bytes)


## 1A. Upload pipeline source files to S3

Uploads preprocess.py, train.py, and inference.py to the team's pipeline
source prefix (the same prefix Notebook 03 uses), then downloads them into
a local pipeline_src/ folder that the pipeline step definitions below
reference.


In [8]:
s3_client = boto3.client("s3")

FILES_TO_UPLOAD = ["preprocess.py", "train.py", "inference.py"]

for filename in FILES_TO_UPLOAD:
    local_path = Path("src") / filename
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"

    if not local_path.exists():
        raise FileNotFoundError(f"Missing local source file: {local_path}")

    s3_client.upload_file(str(local_path), BUCKET, s3_key)
    print(f"Uploaded {local_path} -> s3://{BUCKET}/{s3_key}")

print("Pipeline source files uploaded to:", SCRIPTS_S3_URI)

import shutil

local_src = Path(LOCAL_PIPELINE_SRC)

if local_src.exists():
    shutil.rmtree(local_src)

local_src.mkdir(parents=True, exist_ok=True)

for filename in FILES_TO_UPLOAD:
    s3_key = f"{SCRIPTS_S3_PREFIX}/{filename}"
    local_path = local_src / filename

    s3_client.download_file(BUCKET, s3_key, str(local_path))
    print(f"Downloaded s3://{BUCKET}/{s3_key} -> {local_path}")

print("Downloaded files:")
for p in sorted(local_src.iterdir()):
    print("-", p)

Uploaded src/preprocess.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/preprocess.py


Uploaded src/train.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/train.py
Uploaded src/inference.py -> s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/inference.py
Pipeline source files uploaded to: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src
Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/preprocess.py -> pipeline_src/preprocess.py
Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/train.py -> pipeline_src/train.py


Downloaded s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/pipeline_src/inference.py -> pipeline_src/inference.py
Downloaded files:
- pipeline_src/inference.py
- pipeline_src/preprocess.py
- pipeline_src/train.py


## 2. Define the Triggered SageMaker Pipeline

Same four building blocks as Notebook 03 (Processing -> Training -> Condition -> Register), with one change: ProcessingStep's input source is now a pipeline parameter (InputDataUrl) instead of a hardcoded path, so whichever CSV triggered the run gets trained on. The quality gate and registration step are unchanged.


In [9]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger, ParameterString
from sagemaker.workflow.model_step import ModelStep
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.model import Model
from sagemaker.workflow.pipeline_context import PipelineSession

# Keeps every SDK-managed upload inside the team's own S3 prefix.
pipeline_session = PipelineSession(default_bucket=BUCKET, default_bucket_prefix=PREFIX)

# Pipeline parameters
p_input_data = ParameterString(name='InputDataUrl', default_value=RAW_DATA_URI)
p_c = ParameterFloat(name='C', default_value=10.0)
p_penalty = ParameterString(name='Penalty', default_value='l2')
p_solver = ParameterString(name='Solver', default_value='lbfgs') 
p_max_iter = ParameterInteger(name='MaxIter', default_value=1000) 
p_gate = ParameterFloat(name='QualityGateAUC', default_value=QUALITY_GATE_AUC)

In [10]:
# ProcessingStep trains on whichever file triggered this run.
processor = SKLearnProcessor(
    framework_version='1.2-1', instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1, role=role, sagemaker_session=pipeline_session,
    base_job_name=f'iti113-{TEAM_ID}-{STUDENT_ID}-process-triggered')

step_process = ProcessingStep(
    name='PreprocessData',
    processor=processor,
    inputs=[ProcessingInput(source=p_input_data,
                            destination='/opt/ml/processing/input')],
    outputs=[ProcessingOutput(output_name='processed',
                              source='/opt/ml/processing/output',
                              destination=f'{PIPELINE_ROOT}/processed')],
    code=f'{LOCAL_PIPELINE_SRC}/preprocess.py',
    job_arguments=['--test-size','0.2','--random-state','42']
)
print('ProcessingStep (PreprocessData) defined.')

ProcessingStep (PreprocessData) defined.


In [11]:
estimator = SKLearn(
    entry_point="train.py",
    source_dir=LOCAL_PIPELINE_SRC,
    framework_version="1.2-1",
    instance_type=TRAINING_INSTANCE_TYPE,
    instance_count=1,
    role=role,
    base_job_name=f"iti113-{TEAM_ID}-{STUDENT_ID}-train-triggered",
    sagemaker_session=pipeline_session,
    hyperparameters={
        "reg-c": p_c, 
        "penalty": p_penalty,
        "solver": p_solver,
        "max-iter": p_max_iter,
        "random-state": 42,
        "team-id": TEAM_ID,
        "student-id": STUDENT_ID,
        "semester": SEMESTER,
        "run-name": "triggered_pipeline_run",
    },
    environment={
        "TEAM_ID": TEAM_ID,
        "STUDENT_ID": STUDENT_ID,
        "SEMESTER": SEMESTER,
    },
    metric_definitions=[
        {"Name": "test_auc_roc",   "Regex": "Test AUC-ROC: ([0-9\\.]+)"},
        {"Name": "test_accuracy",  "Regex": "test_accuracy: ([0-9\\.]+)"},
        {"Name": "test_f1",        "Regex": "test_f1: ([0-9\\.]+)"},
        {"Name": "test_precision", "Regex": "test_precision: ([0-9\\.]+)"},
        {"Name": "test_recall",    "Regex": "test_recall: ([0-9\\.]+)"},
    ],
    tags=[
        {"Key": "Course", "Value": "ITI113"},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "Team", "Value": TEAM_ID},
        {"Key": "Student", "Value": STUDENT_ID},
    ],
)

processed_uri = step_process.properties.ProcessingOutputConfig.Outputs["processed"].S3Output.S3Uri

step_train = TrainingStep(
    name="TrainModel",
    estimator=estimator,
    inputs={
        # No content_type: output mixes .npz and .csv; train.py reads each file directly.
        "train": sagemaker.inputs.TrainingInput(s3_data=processed_uri),
        "test": sagemaker.inputs.TrainingInput(s3_data=processed_uri),
    },
)

print("TrainingStep (TrainModel) defined.")

TrainingStep (TrainModel) defined.


In [12]:
# Registers to the shared Model Registry group, so manual and triggered runs share one version history.
model = Model(
    image_uri=estimator.training_image_uri(region),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role,
    entry_point='inference.py',
    source_dir=LOCAL_PIPELINE_SRC
)
step_register = ModelStep(
    name='RegisterModel',
    step_args=model.register(
        content_types=['application/json'],
        response_types=['application/json'],
        inference_instances=['ml.m5.large'],
        transform_instances=['ml.m5.large'],
        model_package_group_name=MODEL_PACKAGE_GROUP,
        approval_status='PendingManualApproval',
    )
)
print('ModelStep (RegisterModel) defined.')

ModelStep (RegisterModel) defined.


In [13]:
# ConditionStep gates on the test AUC-ROC metric captured directly from the training step.
condition = ConditionGreaterThanOrEqualTo(
    left=step_train.properties.FinalMetricDataList["test_auc_roc"].Value,
    right=p_gate
)

step_condition = ConditionStep(
    name="AUCQualityGate",
    conditions=[condition],
    if_steps=[step_register],
    else_steps=[]
)
print("ConditionStep (AUCQualityGate) defined.")

ConditionStep (AUCQualityGate) defined.


In [14]:
# Assemble and upsert the triggered pipeline
pipeline_triggered = Pipeline(
    name=TRIGGERED_PIPELINE_NAME,
    parameters=[p_input_data, p_c, p_penalty, p_solver, p_max_iter, p_gate],
    steps=[step_process, step_train, step_condition],
    sagemaker_session=pipeline_session
)
pipeline_triggered.upsert(role_arn=role)
print(f'Pipeline "{TRIGGERED_PIPELINE_NAME}" upserted.\nView in SageMaker Studio: left sidebar -> Pipelines')


Pipeline "iti113-team03-crypto-scam-detector-triggered" upserted.
View in SageMaker Studio: left sidebar -> Pipelines


## 3. Prove the Trigger Works End-to-End

Notebook stand-in for the Lambda logic: checks the watched S3 prefix for new files, and starts pipeline_triggered with InputDataUrl set to each one.

First, baseline the watch folder so pre-existing files aren't treated as new.


In [15]:
def list_trigger_keys():
    s3 = boto3.client('s3')
    keys = set()
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix=f'{TRIGGER_INCOMING_PREFIX}/'):
        for obj in page.get('Contents', []):
            if obj['Key'].endswith('.csv'):
                keys.add(obj['Key'])
    return keys

def load_seen_keys():
    if SEEN_STATE_FILE.exists():
        return set(json.loads(SEEN_STATE_FILE.read_text()))
    return set()

def save_seen_keys(keys):
    SEEN_STATE_FILE.write_text(json.dumps(sorted(keys)))

# Baseline: anything already in the folder counts as "already seen".
seen_keys = load_seen_keys() | list_trigger_keys()
save_seen_keys(seen_keys)
print(
    f'Baseline established: {len(seen_keys)} existing file(s) in {TRIGGER_INCOMING_URI}/\n'
    f'Any file uploaded after this point will be treated as a new trigger.'
)

Baseline established: 36 existing file(s) in s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming/
Any file uploaded after this point will be treated as a new trigger.


### 3a. Trigger a Retraining Run with a New Raw Data Batch

Uploads a copy of the current raw dataset version (`raw/crypto_scam_dataset_v4.csv`,
10,466 rows) to the watch folder under a new timestamped filename, simulating a fresh
batch of scraped messages arriving.

Raw data is used deliberately: the pipeline's first step is `preprocess.py`, which does
its own cleaning, feature engineering and train/test split, so the trigger folder must
receive data in the same shape as the original source (`id, platform, text, label`)
rather than Notebook 01's already-processed output. The goal of Section 3 is to prove
the trigger mechanism runs end-to-end, not to produce a better model.

In [16]:
import io
from datetime import datetime

RAW_DATA_KEY = f"{PREFIX}/raw/crypto_scam_dataset_v4.csv"   # replaced to use raw data, not processed

s3_client = boto3.client("s3")
raw_obj = s3_client.get_object(Bucket=BUCKET, Key=RAW_DATA_KEY)
raw_bytes = raw_obj["Body"].read()

# Use current timestamp to make filename unique
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
retrain_key = f"{TRIGGER_INCOMING_PREFIX}/retrain_batch_{timestamp}.csv"

s3_client.put_object(Bucket=BUCKET, Key=retrain_key, Body=raw_bytes)

retrain_uri = f"s3://{BUCKET}/{retrain_key}"
print(
    f"Uploaded new batch: {retrain_uri}\n"
    f"Source: s3://{BUCKET}/{RAW_DATA_KEY} ({len(raw_bytes)} bytes)"
)

Uploaded new batch: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming/retrain_batch_20260815-091121.csv
Source: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/raw/crypto_scam_dataset_v4.csv (2229056 bytes)


### 3b. Check the Watch Folder and Trigger a Run

The function a Lambda would run in production. Here it's a notebook cell: list the folder, diff against what's been seen, and start the triggered pipeline for each new file.


In [17]:
def check_trigger_folder_once(auto_start=True):
    """Check the watch folder for files not seen before. For each new file,
    optionally start pipeline_triggered with InputDataUrl set to that file.
    Returns the list of newly detected S3 URIs."""
    current_keys = list_trigger_keys()
    seen = load_seen_keys()
    new_keys = sorted(current_keys - seen)

    if not new_keys:
        print("No new files detected.")
        return []

    new_uris = []
    for key in new_keys:
        uri = f"s3://{BUCKET}/{key}"
        print(f"New file detected: {uri}")
        new_uris.append(uri)

        if auto_start:
            execution = pipeline_triggered.start(parameters={
                'InputDataUrl': uri,
                'C': 10.0, 'Penalty': 'l2', 'Solver': 'lbfgs', 'MaxIter': 1000,
                'QualityGateAUC': QUALITY_GATE_AUC,
            })
            print(f"  -> Started execution: {execution.arn}")

    save_seen_keys(seen | current_keys)
    return new_uris

detected = check_trigger_folder_once(auto_start=True)
if detected:
    print(f"\n{len(detected)} new file(s) triggered a pipeline execution.")


New file detected: s3://nyp-26s1-iti113/iti113/team03/data/crypto-scam-detector/trigger/incoming/retrain_batch_20260815-091121.csv


  -> Started execution: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector-triggered/execution/71ow9tlt0oun

1 new file(s) triggered a pipeline execution.


## 4. Monitor the Triggered Pipeline Execution

Checks the latest execution's status, waits if still running, then confirms whether RegisterModel ran and a new model version appears in the shared Model Registry group.


In [18]:
sm = boto3.client('sagemaker', region_name=region)

import time

TERMINAL = {"Succeeded", "Failed", "Stopped"}
POLL_SECONDS = 30
MAX_MINUTES = 25

result = pipeline_triggered.list_executions()
summaries = result.get("PipelineExecutionSummaries", []) if isinstance(result, dict) else result

if not summaries:
    print("No executions found yet.")
else:
    exec_arn = summaries[0]["PipelineExecutionArn"]
    print(f"Watching: {exec_arn}\n")

    deadline = time.time() + MAX_MINUTES * 60
    seen = {}
    while True:
        status = sm.describe_pipeline_execution(
            PipelineExecutionArn=exec_arn
        )["PipelineExecutionStatus"]

        steps = sm.list_pipeline_execution_steps(PipelineExecutionArn=exec_arn)
        for s in reversed(steps.get("PipelineExecutionSteps", [])):
            name, st = s.get("StepName"), s.get("StepStatus")
            if seen.get(name) != st:
                print(f"  {name}: {st}")
                if s.get("FailureReason"):
                    print(f"     FailureReason: {s['FailureReason']}")
                seen[name] = st

        if status in TERMINAL:
            print(f"\nPipeline {status}.")
            break
        if time.time() > deadline:
            print(f"\nStill {status} after {MAX_MINUTES} min — stopping the watch. "
                  f"The pipeline keeps running; re-run this cell to resume watching.")
            break
        time.sleep(POLL_SECONDS)

Watching: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector-triggered/execution/71ow9tlt0oun



  PreprocessData: Executing


  PreprocessData: Succeeded
  TrainModel: Executing


  TrainModel: Succeeded
  AUCQualityGate: Succeeded
  RegisterModel-RepackModel-0: Executing


  RegisterModel-RepackModel-0: Succeeded
  RegisterModel-RegisterModel: Succeeded

Pipeline Succeeded.


In [19]:
result = pipeline_triggered.list_executions()

# Handle both possible return shapes from list_executions() across SageMaker SDK versions.
if isinstance(result, dict):
    summaries = result.get("PipelineExecutionSummaries", [])
else:
    summaries = result

if summaries:
    latest = summaries[0]
    exec_arn = latest['PipelineExecutionArn']
    print(f"Latest execution: {exec_arn}\nStatus: {latest['PipelineExecutionStatus']}")

    steps_resp = sm.list_pipeline_execution_steps(PipelineExecutionArn=exec_arn)
    print("\nStep-by-step status:")
    for step in steps_resp.get('PipelineExecutionSteps', []):
        print(f"{step.get('StepName')}: {step.get('StepStatus')}")
        failure_reason = step.get('FailureReason')
        if failure_reason:
            print(f"FailureReason: {failure_reason}")
else:
    print("No executions found yet for this pipeline.")

# Show the most recent model package
response = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy='CreationTime',
    SortOrder='Descending',
    MaxResults=1,
)

packages = response['ModelPackageSummaryList']
if not packages:
    print(f"No model packages found yet in {MODEL_PACKAGE_GROUP}.")
else:
    pkg = packages[0]
    print(
        f"Most recent model in {MODEL_PACKAGE_GROUP}:\n"
        f"Model version: {pkg['ModelPackageVersion']}\n"
        f"Status: {pkg['ModelPackageStatus']}\n"
        f"Approval status: {pkg['ModelApprovalStatus']}"
    )


Latest execution: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/iti113-team03-crypto-scam-detector-triggered/execution/71ow9tlt0oun
Status: Succeeded

Step-by-step status:
RegisterModel-RegisterModel: Succeeded
RegisterModel-RepackModel-0: Succeeded
AUCQualityGate: Succeeded
TrainModel: Succeeded
PreprocessData: Succeeded
Most recent model in team03-CryptoScamDetector:
Model version: 36
Status: Completed
Approval status: PendingManualApproval


## 5. How This Maps to a Production Setup

Maps each mechanism in this notebook to its production CI/CD equivalent.

| Production component | This notebook's stand-in |
|---|---|
| S3 PutObject event | A new .csv appearing under trigger/incoming/ |
| EventBridge rule (watches the S3 event) | The polling loop / check_trigger_folder_once() call in section 3-4 |
| Lambda function (calls start_pipeline_execution) | pipeline_triggered.start(parameters={'InputDataUrl': ...}) inside check_trigger_folder_once() |
| CloudWatch Logs (execution history) | The notebook's own print() output, plus pipeline_triggered.list_executions() |
| IAM role restricting Lambda's S3/SageMaker access | The same team-scoped SageMaker execution role this notebook already runs under |
